# Banking Portfolio Credit Risk Analytics - Exploratory Data Analysis (EDA)

## 1. Business Context & Problem Statement
Financial institutions face significant challenges in accurately assessing customer credit risk while balancing loan growth and minimizing defaults. Existing lending decisions often rely on multiple risk indicators that require effective analysis of customer transaction and financial data. 

This project aims to develop a data-driven credit risk analytics solution that improves risk assessment, predicts the likelihood of default, and supports informed credit decision-making. 

Through this notebook, we perform a robust Exploratory Data Analysis (EDA) on the customer demographics, balance portfolios, retirement accounts, and outstanding debts to uncover structural correlations, liquidity indicators, and risk weight concentrations.

In [ ]:
# Environment Setup & Library Ingestion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling for clean, professional visualization design
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 10
import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading & Profiling
We ingest the primary client-banking repository dataset containing transaction, demographic, and financial details of 3,000 banking clients.

In [ ]:
# Load the raw client-banking datasets
data_path = '../data/raw/banking-clients.csv'
df = pd.read_csv(data_path)

# Inspect dataset dimensions
print(f"Dataset Schema Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

# Concise overview of datatypes and non-null elements
df.info()

In [ ]:
# Display first 5 client records
df.head()

## 3. Data Cleaning & Feature Engineering
We align datatypes (converting text-based dates to datetime) and construct derived indicators matching the core business requirements:
- **Income Band**: Discretizes `Estimated Income` into Low (<$100K), Mid ($100K - $300K), and High (>$300K) tiers.
- **Processing Fees**: Calculates implied fees based on the loan size and client fee structure.

In [ ]:
# 1. Convert Joined Bank date string to Pandas datetime format
df['Joined Bank'] = pd.to_datetime(df['Joined Bank'], format='%d-%m-%Y')

# 2. Income discretization using pd.cut
income_bins = [0, 100000, 300000, float('inf')]
income_labels = ['Low', 'Mid', 'High']
df['Income Band'] = pd.cut(df['Estimated Income'], bins=income_bins, labels=income_labels, include_lowest=True)

# 3. Implied Processing Fee calculation (based on Fee Structure and Total Loans)
# High fee tier: 5%, Mid: 3%, Low: 1%
df['Processing Fee Rate'] = df['Fee Structure'].map({'High': 0.05, 'Mid': 0.03, 'Low': 0.01}).fillna(0.00)
df['Total Loan Amount'] = df['Bank Loans'] + df['Business Lending'] + df['Credit Card Balance']
df['Calculated Processing Fees'] = df['Total Loan Amount'] * df['Processing Fee Rate']

# Verify feature engineering results
df[['Client ID', 'Estimated Income', 'Income Band', 'Fee Structure', 'Total Loan Amount', 'Calculated Processing Fees']].head()

## 4. Statistical Profiling & Descriptive Statistics

In [ ]:
# Descriptive stats for numerical features
df.describe().T

In [ ]:
# Verify there are no missing values in the primary columns
df.isnull().sum()

## 5. Exploratory Visualizations & Regional Risk Distribution
### Univariate Analysis: Numerical Feature Distributions

In [ ]:
# Distribution of key financial assets and balance metrics
numerical_cols = ['Age', 'Estimated Income', 'Superannuation Savings', 'Credit Card Balance', 
                  'Bank Loans', 'Bank Deposits', 'Checking Accounts', 'Saving Accounts', 
                  'Foreign Currency Account', 'Business Lending']

plt.figure(figsize=(16, 12))
for i, col in enumerate(numerical_cols):
    plt.subplot(4, 3, i + 1)
    sns.histplot(df[col], kde=True, color='teal')
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

### Bivariate Analysis: Categorical Feature Frequencies

In [ ]:
# Check representation of segments across categoricals
categorical_cols = ['Risk Weighting', 'Nationality', 'Fee Structure', 'Loyalty Classification', 'Income Band']

plt.figure(figsize=(16, 10))
for i, col in enumerate(categorical_cols):
    plt.subplot(2, 3, i + 1)
    sns.countplot(data=df, x=col, order=df[col].value_counts().index, palette='viridis')
    plt.title(f'Client Share by {col}')
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Correlation Analysis
Determining structural dependencies across client balances, assets, and liabilities.

In [ ]:
# Select numerical variables for correlation matrix
corr_cols = ['Age', 'Estimated Income', 'Superannuation Savings', 'Credit Card Balance',
             'Bank Loans', 'Bank Deposits', 'Checking Accounts', 'Saving Accounts',
             'Foreign Currency Account', 'Business Lending', 'Properties Owned']

correlation_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Customer Portfolios', fontsize=16)
plt.show()

### Key Portfolio Relationships

In [ ]:
# Deep dive regression plots mapping key asset dependencies
pairs_to_plot = [
    ('Bank Deposits', 'Saving Accounts'),
    ('Checking Accounts', 'Saving Accounts'),
    ('Estimated Income', 'Checking Accounts'),
    ('Bank Loans', 'Credit Card Balance'),
    ('Business Lending', 'Bank Loans')
]

for x_col, y_col in pairs_to_plot:
    plt.figure(figsize=(8, 5))
    sns.regplot(
        data=df,
        x=x_col,
        y=y_col,
        scatter_kws={'alpha': 0.3, 'color': 'navy'},
        line_kws={'color': 'red', 'linewidth': 2}
    )
    plt.title(f'Dependency Analysis: {x_col} vs {y_col}', fontsize=12)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.tight_layout()
    plt.show()

## 6. Mathematical Risk and Liquidity Formulations
To support technical credit decisioning, we formalize the credit assessment indicators:

1. **Aggregate Debt Exposure (\(D_{total}\))**:
   $$D_{total} = L_{bank} + L_{business} + B_{cc}$$
   Where \(L_{bank}\) is Bank Loans, \(L_{business}\) is Business Lending, and \(B_{cc}\) is Credit Card Balance.

2. **Total Deposit Liquidity (\(S_{total}\))**:
   $$S_{total} = D_{bank} + A_{saving} + A_{checking} + A_{fx}$$
   Where \(D_{bank}\) is Bank Deposits, \(A_{saving}\) is Savings accounts, \(A_{checking}\) is Checking accounts, and \(A_{fx}\) is Foreign Currency accounts.

3. **Credit Leverage Index (\(CLI\))**:
   $$CLI = \frac{D_{total}}{S_{total}}$$
   *A higher CLI represents elevated default risk as credit exposure exceeds liquid deposit collateral.*

## 7. Strategic Insights Derived from EDA

### A. Deposits and Savings Over-Liquidity
The strong positive correlation (\(r \approx 0.80\)) between `Bank Deposits` and `Saving Accounts` indicates a high volume of unallocated cash (capital inefficacy). High-net-worth clients hold massive liquid reserves in basic deposit vehicles rather than higher-yielding products. This represents a significant opportunity for the bank to cross-sell wealth management products (AUM) via the Investment Advisors.

### B. Credit Leverage Concentration
Mid-income earners ($100K - $300K) exhibit the highest volume of credit exposure. However, high-income earners hold negligible traditional loan structures. This concentration of loans in the middle-market represents a critical systemic risk for the credit portfolio, particularly under high-interest rate or inflation environments where the default probability of mid-income brackets spikes.

### C. Risk-Weighting and Property Dynamics
Property ownership shows negligible correlation with traditional risk weightings and deposit values. This suggests that the current internal credit scoring and risk weighting models do not heavily weight real-estate collateral, indicating a potential under-collateralization in the higher risk brackets (Risk Tiers 4 and 5).